# Silver to Gold — CineData Analytics
Modelagem dimensional (Star Schema), tabela de contexto para o RAG do time de IA, e
respostas às 6 perguntas de negócio (Desafio de Analytics, seção 4 do enunciado).

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src.gold.genai_context import agregar_pessoas_por_filme, construir_genai_context
from src.gold.star_schema import (
    construir_bridge_movie_company,
    construir_bridge_movie_genre,
    construir_bridge_movie_person,
    construir_dim_companies,
    construir_dim_genres,
    construir_dim_movies,
    construir_dim_people,
    construir_dim_reviews,
    construir_fact_movies_performance,
)

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

**Regras do Star Schema:** chaves substitutas (`sk_*`) geradas com `row_number()`; `id_filme` é mantido como chave natural (STRING). Dimensões de pessoas e produtoras são separadas, e gêneros, pessoas e produtoras se ligam ao filme por **tabelas-ponte** (relação N:N), o que evita duplicar linhas da fato.

In [ ]:
# Dimensoes base
df_filmes = spark.table("silver.tb_info_filmes")
df_generos = spark.table("silver.tb_generos")
df_pessoas_empresas = spark.table("silver.tb_pessoas_empresas")
df_financeiro = spark.table("silver.tb_financeiro_filmes")
df_engajamento = spark.table("silver.tb_metricas_engajamento")
df_avaliacoes = spark.table("silver.tb_avaliacoes_usuarios")

dim_movies = construir_dim_movies(df_filmes)
dim_genres = construir_dim_genres(df_generos)
dim_people = construir_dim_people(df_pessoas_empresas)
dim_companies = construir_dim_companies(df_pessoas_empresas)

dim_movies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_movies")
dim_genres.write.format("delta").mode("overwrite").saveAsTable("gold.dim_genres")
dim_people.write.format("delta").mode("overwrite").saveAsTable("gold.dim_people")
dim_companies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_companies")

**Regras da fato:** um registro por **filme lançado** (`status_filme = "Lançado"`), com métricas financeiras (`DECIMAL(18,2)`) e de engajamento. Como as tabelas de origem já chegam deduplicadas da Silver e os relacionamentos N:N ficam nas bridges, os joins não multiplicam linhas.

In [ ]:
# Fato + bridges + dim_reviews
fact = construir_fact_movies_performance(dim_movies, df_financeiro, df_engajamento)
bridge_genre = construir_bridge_movie_genre(dim_movies, dim_genres, df_generos)
bridge_person = construir_bridge_movie_person(dim_movies, dim_people, df_pessoas_empresas)
bridge_company = construir_bridge_movie_company(dim_movies, dim_companies, df_pessoas_empresas)
dim_reviews = construir_dim_reviews(dim_movies, df_avaliacoes)

fact.write.format("delta").mode("overwrite").saveAsTable("gold.fact_movies_performance")
bridge_genre.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_genre")
bridge_person.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_person")
bridge_company.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_company")
dim_reviews.write.format("delta").mode("overwrite").saveAsTable("gold.dim_reviews")

display(fact)

**Regras da tabela de contexto para IA:** o documento é uma frase corrida montada a partir da fato e das dimensões. Como `concat` devolve `NULL` se qualquer campo for nulo (o que faria o filme sumir da tabela), **todo campo que pode faltar** (receita, orçamento, elenco, diretor, sinopse, ano, título) passa por `coalesce` com um texto de fallback antes da concatenação. Elenco e diretor são agregados em uma só string, em ordem alfabética para o texto ser estável entre execuções.

In [ ]:
# gold_genai_movies_context -- tabela de contexto para o time de IA (Vector Search / RAG)
atores_por_filme = agregar_pessoas_por_filme(bridge_person, dim_people, "Ator")
diretores_por_filme = agregar_pessoas_por_filme(bridge_person, dim_people, "Diretor")

contexto_genai = construir_genai_context(dim_movies, fact, atores_por_filme, diretores_por_filme)
contexto_genai.write.format("delta").mode("overwrite").saveAsTable("gold.gold_genai_movies_context")

display(contexto_genai.limit(5))

## Desafio de Analytics

**Perguntas 5 e 6:** a "data limite" é o lançamento mais recente **já ocorrido** de um filme "Lançado" (ignora datas futuras e não lançados, como o enunciado pede; a base tem um filme "Lançado" com data em 2029). Os recortes de 2 e 5 anos contam para trás a partir dela.

In [ ]:
# 1. Receita total (em R$) de todos os filmes da base
display(fact.agg(F.sum("receita_brl").alias("receita_total_brl")))

In [ ]:
# 2. Os 5 filmes com maior popularidade (titulo + popularidade)
display(
    fact.join(dim_movies, "sk_movie_id")
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc())
    .limit(5)
)

In [ ]:
# 3. Quantidade de filmes por genero, do maior para o menor volume
display(
    bridge_genre.join(dim_genres, "sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.count("*").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc())
)

In [ ]:
# 4. Os 10 filmes de maior receita: titulo, receita (US$ e R$) e posicao no ranking
janela_receita = Window.orderBy(F.col("receita_usd").desc())
display(
    fact.join(dim_movies, "sk_movie_id")
    .select("titulo", "receita_usd", "receita_brl")
    .withColumn("posicao_ranking", F.rank().over(janela_receita))
    .orderBy("posicao_ranking")
    .limit(10)
)

In [ ]:
# 5. Ator com mais participacoes nos filmes lancados nos ultimos 2 anos
# (data limite = lancamento mais recente da base, ignorando datas futuras/nao lancadas)
# Ignora datas futuras e filmes nao lancados (a base tem "Lancado" com data em 2029).
data_limite = (
    dim_movies.filter(
        (F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date())
    )
    .agg(F.max("data_lancamento"))
    .first()[0]
)
print(f"Data limite (lancamento mais recente ja ocorrido): {data_limite}")
data_corte_2_anos = F.add_months(F.lit(data_limite), -24)

filmes_recentes = dim_movies.filter(
    (F.col("data_lancamento") <= F.lit(data_limite))
    & (F.col("data_lancamento") >= data_corte_2_anos)
)
display(
    bridge_person.join(filmes_recentes, "sk_movie_id")
    .join(dim_people.filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .groupBy("nome_pessoa")
    .agg(F.count("*").alias("qtd_participacoes"))
    .orderBy(F.col("qtd_participacoes").desc())
    .limit(1)
)

In [ ]:
# 6. Produtora com maior lucro nos ultimos 5 anos
data_corte_5_anos = F.add_months(F.lit(data_limite), -60)
filmes_ultimos_5_anos = dim_movies.filter(
    (F.col("data_lancamento") <= F.lit(data_limite))
    & (F.col("data_lancamento") >= data_corte_5_anos)
)
display(
    bridge_company.join(filmes_ultimos_5_anos, "sk_movie_id")
    .join(fact, "sk_movie_id")
    .join(dim_companies, "sk_company_id")
    .where(F.col("lucro_usd").isNotNull())
    .groupBy("nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"))
    .orderBy(F.col("lucro_total_usd").desc())
    .limit(1)
)